# V2 push, step 1 — the data v1 factory (DeepSeek self-breaking)

**Runtime: CPU** (API calls + test execution — no GPU, near-zero units).
~45–90 min, ~$2–4 of DeepSeek credit. Safe to run in parallel with notebook 10.

**Why (literature pass 4, S2.17):** our exam gap is out-of-distribution — the
exam has docstring-rich functions with human-crafted, often multi-line bugs;
our v0 corpus is docstring-free single-operator AST mutations. QiMeng-PRepair's
'Self-Breaking' and Repair-R1's LLM-generated defects both show richer bugs are
the lever. This notebook, per gold function:
1. DeepSeek adds a **concise docstring** (closes the format gap) — the
   docstringed function must still PASS its full MBPP+ suite
2. DeepSeek writes **6 diverse buggy variants** (3 subtle single-line +
   3 harder multi-line/compound), each labeled with our taxonomy
3. Every bug is **execution-certified**: compiles AND fails ≥1 test of the
   full suite (timeout counts as a bug); deduped by normalized AST

**Invariants preserved:** each function keeps its **v0 split** (never re-hash —
the docstring changes the code); source pool is already contamination-screened
(v0); nothing HumanEval-derived. Output: `data_v1_bugs.json` = v0 bugs + new
certified bugs (~2.5–2.9k total) + docstringed restraint suite, saved to Drive
`phase8/`. Checkpoint-resume built in.

In [ ]:
# Setup: Drive, repo, the gold pool (v0 restraint = all gold functions + suites)
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, ast, random, importlib
PHASE8 = '/content/drive/MyDrive/rl-post-training/phase8'
os.makedirs(PHASE8, exist_ok=True)
!rm -rf /content/ptl
!git clone -q https://github.com/nidhi1603/post-training-lab.git /content/ptl
sys.path.insert(0, '/content/ptl/src')
ver = !git -C /content/ptl log --oneline -1
print('FACTORY VERSION:', ver[0])
v0_bugs = json.load(open('/content/ptl/data/data_v0_bugs.json'))
gold = json.load(open('/content/ptl/data/data_v0_restraint.json'))
print(len(gold), 'gold functions (pre-screened, MBPP+ suites) |', len(v0_bugs), 'v0 bugs to carry over')

In [ ]:
%%capture
!pip install -q openai

In [ ]:
# DeepSeek client (key from Colab Secrets ONLY) + auth ping
from google.colab import userdata
from openai import OpenAI
client = OpenAI(api_key=userdata.get('DEEPSEEK_API_KEY'),
                base_url='https://api.deepseek.com')
ping = client.chat.completions.create(model='deepseek-chat', max_tokens=5,
    messages=[{'role': 'user', 'content': 'Reply with exactly: OK'}])
print('Auth OK:', ping.choices[0].message.content.strip())

In [ ]:
# Certification gate (v0 rules: compile + fail >=1 of FULL suite; timeout = bug)
import subprocess, tempfile
from concurrent.futures import ThreadPoolExecutor

def run_with_tests(source, rec, timeout=6):
    prog = '\n'.join(list(rec['test_imports']) + [source] + list(rec['test_list']))
    with tempfile.NamedTemporaryFile('w', suffix='.py', delete=False) as f:
        f.write(prog); path = f.name
    try:
        r = subprocess.run([sys.executable, path], capture_output=True, timeout=timeout)
        return 'pass' if r.returncode == 0 else 'fail'
    except subprocess.TimeoutExpired:
        return 'timeout'
    finally:
        os.unlink(path)

def normalize(src):
    try:
        return ast.unparse(ast.parse(src))
    except SyntaxError:
        return None

CATS = {'variable_misuse', 'operator_misuse', 'value_misuse',
        'function_misuse', 'missing_logic', 'excess_logic', 'compound'}

PROMPT_TMPL = """You are given a correct Python function and its test suite.

Correct function:
```python
{code}
```

Some of its tests:
```python
{tests}
```

Task 1: rewrite the function UNCHANGED in logic, only adding a concise 1-3 line
docstring describing what it does. Do not rename anything or alter behavior.

Task 2: create 6 DISTINCT buggy variants of that docstringed function.
Variants 1-3: subtle single-line bugs (wrong operator, off-by-one, wrong
variable, wrong constant...). Variants 4-6: harder multi-line or compound
semantic bugs (wrong loop structure, missing case, mishandled edge condition,
two interacting mistakes...). Each must be a REALISTIC programmer mistake,
must still be syntactically valid, and must produce wrong behavior on at least
one test. Keep the docstring in every variant. Label each with exactly one
category from: variable_misuse, operator_misuse, value_misuse, function_misuse,
missing_logic, excess_logic, compound.

Return STRICT JSON only, no markdown fences:
{{"fixed": "<full docstringed function>", "bugs": [{{"category": "...", "description": "<one line>", "code": "<full buggy function>"}}]}}"""

def parse_json(text):
    t = text.strip()
    if t.startswith('```'):
        t = t.split('```')[1]
        if t.startswith('json'):
            t = t[4:]
    return json.loads(t)

def break_function(rec, temps=(0.7, 1.0)):
    """Returns (docstringed_fixed, certified_bugs, usage) — bugs certified on FULL suite."""
    tin = tout = 0
    shown = '\n'.join(list(rec['test_list'])[:8])
    for t in temps:
        try:
            r = client.chat.completions.create(model='deepseek-chat', temperature=t,
                max_tokens=3000, messages=[{'role': 'user', 'content':
                    PROMPT_TMPL.format(code=rec['code'], tests=shown)}])
            tin += r.usage.prompt_tokens; tout += r.usage.completion_tokens
            obj = parse_json(r.choices[0].message.content)
            fixed = obj['fixed']
        except Exception:
            continue
        if run_with_tests(fixed, rec) != 'pass':
            fixed = rec['code']   # docstring broke it -> fall back, bugs vs original
        seen = {normalize(fixed)}
        certified = []
        for i, b in enumerate(obj.get('bugs', [])):
            code, cat = b.get('code', ''), b.get('category', 'compound')
            if cat not in CATS:
                cat = 'compound'
            norm = normalize(code)
            if not norm or norm in seen:
                continue
            v = run_with_tests(code, rec)
            if v in ('fail', 'timeout'):
                seen.add(norm)
                certified.append({'category': cat, 'tier': 'subtle' if i < 3 else 'compound_tier',
                                  'description': b.get('description', ''), 'code': code})
        if certified:
            return fixed, certified, (tin, tout)
    return rec['code'], [], (tin, tout)

In [ ]:
# SMOKE: 2 functions — eyeball docstrings, bug realism, certification counts
for rec in gold[:2]:
    fixed, bugs, usage = break_function(rec)
    print('=' * 70)
    print(rec['id'], '| certified bugs:', len(bugs))
    print('--- docstringed fixed (head) ---')
    print('\n'.join(fixed.splitlines()[:8]))
    for b in bugs[:2]:
        print(f"--- {b['tier']} / {b['category']}: {b['description']}")
print('\nHealthy = real docstring, realistic diverse bugs, >=3 certified per function.')
print('If bugs look silly (renamed function, deleted body) or 0-1 certified: STOP and report.')

In [ ]:
# FULL RUN with checkpoint-resume (safe to re-run; ~45-90 min for 374 functions)
import time
CKPT = f'{PHASE8}/v1_factory_ckpt.json'
done = json.load(open(CKPT)) if os.path.exists(CKPT) else {}
todo = [r for r in gold if r['id'] not in done]
print(f'resuming: {len(done)} done, {len(todo)} to go')
t0 = time.time()

def work(rec):
    fixed, bugs, (tin, tout) = break_function(rec)
    return rec['id'], {'source_task': rec['source_task'], 'split': rec['split'],
                       'fixed': fixed, 'bugs': bugs,
                       'test_imports': list(rec['test_imports']),
                       'test_list': list(rec['test_list']),
                       'in_tokens': tin, 'out_tokens': tout}

for i in range(0, len(todo), 12):
    batch = todo[i:i+12]
    with ThreadPoolExecutor(max_workers=4) as pool:
        for fid, out in pool.map(work, batch):
            done[fid] = out
    json.dump(done, open(CKPT, 'w'))
    nb = sum(len(d['bugs']) for d in done.values())
    print(f'{len(done)}/{len(gold)} functions | {nb} certified bugs | {time.time()-t0:.0f}s')
print('checkpointed to', CKPT)

In [ ]:
# ASSEMBLE data v1 + THE REPORT
from collections import Counter
new_bugs, restraint_v1 = [], []
for fid, d in done.items():
    restraint_v1.append({'id': fid.replace('-clean', '') + '-clean-v1',
                         'source_task': d['source_task'], 'split': d['split'],
                         'code': d['fixed'], 'test_imports': d['test_imports'],
                         'test_list': d['test_list']})
    for j, b in enumerate(d['bugs']):
        new_bugs.append({'id': f"{d['source_task']}-ds-{b['category']}-{j}",
                         'source_task': d['source_task'], 'split': d['split'],
                         'category': b['category'], 'tier': b['tier'],
                         'gen': 'deepseek_v1', 'description': b['description'],
                         'buggy': b['code'], 'fixed': d['fixed'],
                         'test_imports': d['test_imports'], 'test_list': d['test_list']})
v1_bugs = v0_bugs + new_bugs   # v0 records carried over untouched
json.dump(v1_bugs, open(f'{PHASE8}/data_v1_bugs.json', 'w'), indent=1)
json.dump(restraint_v1, open(f'{PHASE8}/data_v1_restraint.json', 'w'), indent=1)
print(f'=== DATA V1: {len(v1_bugs)} bugs ({len(v0_bugs)} v0 + {len(new_bugs)} new) ===')
print('\nnew bugs by split:', dict(Counter(b['split'] for b in new_bugs)))
print('new bugs by tier:', dict(Counter(b['tier'] for b in new_bugs)))
print('\nnew bugs by category:')
for cat, n in Counter(b['category'] for b in new_bugs).most_common():
    print(f'  {cat:<18} {n:>4}')
funcs_with = sum(1 for d in done.values() if d['bugs'])
print(f'\nfunctions with >=1 certified bug: {funcs_with}/{len(done)}')
print(f'docstringed restraint suite: {len(restraint_v1)}')
tin = sum(d['in_tokens'] for d in done.values())
tout = sum(d['out_tokens'] for d in done.values())
print(f'API usage: {tin:,} in + {tout:,} out ~ ${tin/1e6*0.27 + tout/1e6*1.10:.2f}')
print('\nSaved to Drive phase8/. Bring the whole report back to the session.')

## Bring back to the session
1. The **smoke-cell output** (docstring + bug realism — the human check)
2. The **final report** (totals by tier/category/split, cert rate, cost)

Notes: each function keeps its **v0 split** (no re-hashing), so no train/dev/
heldout leakage between v0 and v1. The gold pool was contamination-screened in
v0; nothing HumanEval-derived enters. After this: SFT v2 on data v1 (same
frozen recipe), then GRPO v2 with the edit-aware penalty.